In [ ]:
import os
import json

from glob import glob

In [ ]:
COCO_CATEGORIES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", 
    "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", 
    "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", 
    "banana", "apple", "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch", "potted plant", "bed", "dining table", "toilet", 
    "tv", "laptop", "mouse", "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors", "teddy bear", 
    "hair drier", "toothbrush"
]

print(f"Total number of COCO categories: {len(COCO_CATEGORIES)}")

In [ ]:
json_files = glob(os.path.join("val", "*.json"))

In [ ]:
for json_file in json_files:
    with open(json_file) as f:
        data = json.load(f)

    per_class_instance_count = {}
    clsId_to_clsName = {cat["id"]: cat["name"] for cat in data["categories"]}

    for annotation in data["annotations"]:
        clsId = annotation["category_id"]
        cls_name = clsId_to_clsName[clsId]
        if cls_name.lower() in COCO_CATEGORIES:
            print(f"Info: Class '{cls_name}' from COCO categories found in {json_file}.")
            per_class_instance_count[cls_name] = per_class_instance_count.get(cls_name, 0) + 1

    print(f"File: {json_file}")
    print(f"Total number of classes: {len(per_class_instance_count)}")
    print("Per-class instance count:")
    for cls_name, count in per_class_instance_count.items():
        print(f"{cls_name}: {count}")
    print("\n")

    

In [ ]:
from collections import defaultdict
from pathlib import Path
import random

In [ ]:
IMAGES_PER_CLASS = 100
RANDOM_SEED = 42
OUTPUT_DIR = Path("val/coco_aligned_subsets")

rng = random.Random(RANDOM_SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def normalise_class_name(name: str) -> str:
    return " ".join(name.strip().lower().split())

In [ ]:
for json_file in json_files:
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    clsId_to_clsName = {
        category["id"]: category["name"]
        for category in data["categories"]
    }

    coco_category_ids = {
        category_id
        for category_id, category_name in clsId_to_clsName.items()
        if normalise_class_name(category_name) in COCO_CATEGORIES
    }

    coco_category_names = {
        category_id: clsId_to_clsName[category_id]
        for category_id in coco_category_ids
    }

    annotations_by_image = defaultdict(list)
    image_ids_by_category = defaultdict(set)

    for annotation in data["annotations"]:
        category_id = annotation["category_id"]

        if category_id not in coco_category_ids:
            continue

        image_id = annotation["image_id"]

        annotations_by_image[image_id].append(annotation)
        image_ids_by_category[category_id].add(image_id)

    print(f"\nSelecting up to {IMAGES_PER_CLASS} images per COCO-overlapping class.")
    print(f"Source annotation file: {json_file}")
    print(f"Matched Objects365 categories: {len(coco_category_ids)}")

    selected_image_ids = set()
    selection_report = []

    for category_id in sorted(coco_category_ids):
        category_name = coco_category_names[category_id]

        candidate_image_ids = sorted(image_ids_by_category[category_id])

        if len(candidate_image_ids) == 0:
            print(f"Warning: no images found for class '{category_name}'.")
            continue

        remaining_needed = max(
            0,
            IMAGES_PER_CLASS - sum(
                1
                for image_id in selected_image_ids
                if image_id in image_ids_by_category[category_id]
            ),
        )

        if remaining_needed > 0:
            available_unselected_ids = [
                image_id
                for image_id in candidate_image_ids
                if image_id not in selected_image_ids
            ]

            rng.shuffle(available_unselected_ids)

            selected_image_ids.update(
                available_unselected_ids[:remaining_needed]
            )

        final_selected_count = len(
            selected_image_ids.intersection(image_ids_by_category[category_id])
        )

        selection_report.append(
            {
                "category_id": category_id,
                "category_name": category_name,
                "available_images": len(candidate_image_ids),
                "selected_images_containing_class": final_selected_count,
                "target_images": IMAGES_PER_CLASS,
                "target_reached": final_selected_count >= IMAGES_PER_CLASS,
            }
        )

    images_by_id = {
        image["id"]: image
        for image in data["images"]
    }

    selected_images = [
        images_by_id[image_id]
        for image_id in sorted(selected_image_ids)
        if image_id in images_by_id
    ]

    selected_annotations = [
        annotation
        for annotation in data["annotations"]
        if annotation["image_id"] in selected_image_ids
        and annotation["category_id"] in coco_category_ids
    ]

    selected_categories = [
        category
        for category in data["categories"]
        if category["id"] in coco_category_ids
    ]

    subset_data = {
        "info": {
            **data.get("info", {}),
            "description": (
                "Objects365 validation subset filtered to COCO-overlapping "
                f"categories, targeting {IMAGES_PER_CLASS} images per class."
            ),
            "selection_seed": RANDOM_SEED,
            "images_per_class_target": IMAGES_PER_CLASS,
            "source_annotation_file": os.path.basename(json_file),
        },
        "licenses": data.get("licenses", []),
        "images": selected_images,
        "annotations": selected_annotations,
        "categories": selected_categories,
    }

    source_stem = Path(json_file).stem
    output_json_path = OUTPUT_DIR / (
        f"{source_stem}_coco_overlap_{IMAGES_PER_CLASS}_images_per_class.json"
    )

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(subset_data, f, ensure_ascii=False, indent=2)

    report_path = OUTPUT_DIR / (
        f"{source_stem}_coco_overlap_{IMAGES_PER_CLASS}_selection_report.json"
    )

    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(selection_report, f, ensure_ascii=False, indent=2)

    print("\nSubset creation completed.")
    print(f"Unique selected images: {len(selected_images)}")
    print(f"Selected annotations: {len(selected_annotations)}")
    print(f"Selected categories: {len(selected_categories)}")
    print(f"COCO-format subset JSON: {output_json_path}")
    print(f"Selection report: {report_path}")

    print("\nPer-class selection summary:")
    for row in selection_report:
        status = "OK" if row["target_reached"] else "INSUFFICIENT"
        print(
            f"[{status}] {row['category_name']}: "
            f"{row['selected_images_containing_class']}/"
            f"{row['target_images']} selected, "
            f"{row['available_images']} available"
        )